# Global Growth Comparison (Multi-language)

This notebook compares traffic trends across different Wikipedia language editions (English, Spanish, French, Japanese, German, Hindi) to identify regional growth patterns.

In [ ]:
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to path
sys.path.append(os.path.abspath('../../src'))
from api_client import WikipediaAPIClient

In [ ]:
client = WikipediaAPIClient()
languages = {
    'en': 'en.wikipedia.org',
    'es': 'es.wikipedia.org',
    'fr': 'fr.wikipedia.org',
    'ja': 'ja.wikipedia.org',
    'de': 'de.wikipedia.org',
    'hi': 'hi.wikipedia.org'
}

start_date = "20180101"
end_date = "20240101"

lang_dfs = []
for lang, proj in languages.items():
    print(f"Fetching {lang} pageviews...")
    df = client.get_aggregate_pageviews(proj, start_date, end_date)
    if df is not None:
        df['language'] = lang
        lang_dfs.append(df)

all_langs_df = pd.concat(lang_dfs, ignore_index=True)
all_langs_df.head()

In [ ]:
# Visualization: Absolute Pageviews
plt.figure(figsize=(14, 8))
sns.lineplot(data=all_langs_df, x='timestamp', y='views', hue='language', linewidth=2)
plt.title('Wikipedia Pageviews Comparison by Language Edition', fontsize=16)
plt.yscale('log')  # Log scale to compare different orders of magnitude
plt.ylabel('Pageviews (Log Scale)')
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.show()

In [ ]:
# Relative Growth Analysis (Normalized to 100 at start)
normalized_dfs = []
for lang in all_langs_df['language'].unique():
    subset = all_langs_df[all_langs_df['language'] == lang].sort_values('timestamp')
    first_val = subset['views'].iloc[0]
    subset['normalized_growth'] = (subset['views'] / first_val) * 100
    normalized_dfs.append(subset)

growth_df = pd.concat(normalized_dfs)

plt.figure(figsize=(14, 8))
sns.lineplot(data=growth_df, x='timestamp', y='normalized_growth', hue='language', linewidth=2)
plt.title('Relative Traffic Growth by Language (Base=100 in 2018)', fontsize=16)
plt.ylabel('Growth Index')
plt.grid(True)
plt.show()